# Benchmark Yambda

In [1]:
!pip install -q gdown implicit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 2.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [24]:
from typing import Dict, List
from collections import defaultdict, Counter
import gc
import math
import os
import random
import subprocess

import numpy as np
import polars as pl
import scipy.sparse as sp
from tqdm.auto import tqdm

from implicit.nearest_neighbours import TFIDFRecommender
from implicit.als import AlternatingLeastSquares
from datasets import load_dataset

In [26]:
DATA_DIR = "."
TOPK = 100
CORE_MIN_INTERACTIONS_PER_ITEM = 5
TEST_INTERVAL_SECONDS = 7 * 24 * 60 * 60
TEMPORAL_THRESHOLD = 25_395_195

def set_random_seed(seed: int = 42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)

set_random_seed(42)

def _first_existing(candidates: List[str]) -> str | None:
    for name in candidates:
        path = os.path.join(DATA_DIR, name)
        if os.path.exists(path):
            return path
    return None

def maybe_download_dataset():
    path_interactions = _first_existing(["interactions.parquet", "likes.parquet", "listens.parquet"])
    path_embeddings = _first_existing(["embeddings.parquet", "item_embeddings.parquet", "track_embeddings.parquet"])

    if path_interactions is not None and path_embeddings is not None:
        return

    subprocess.run(["gdown", "--id", "1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS", "-O", "dataset.zip"], check=True)
    subprocess.run(["unzip", "-qo", "dataset.zip"], check=True)

maybe_download_dataset()


In [27]:
def load_data():
    ds = load_dataset("yandex/yambda", data_dir="flat/5b", data_files="likes.parquet")
    interactions = ds['train'].to_polars()

    core_items = (
        interactions
        .group_by("item_id")
        .agg(pl.len().alias("count"))
        .filter(pl.col("count") >= CORE_MIN_INTERACTIONS_PER_ITEM)
        .select("item_id")
    )

    data = (
        interactions
        .join(core_items, on="item_id", how="semi")
        .sort(["uid", "timestamp"])
    )

    train = data.filter(pl.col("timestamp") < TEMPORAL_THRESHOLD)
    test = data.filter(pl.col("timestamp") >= TEMPORAL_THRESHOLD)

    train_users = train.select("uid").unique()
    test = test.join(train_users, on="uid", how="semi")

    test_targets_df = (
        test
        .select(["uid", "item_id"])
        .unique(subset=["uid", "item_id"], maintain_order=True)
        .group_by("uid", maintain_order=True)
        .agg(pl.col("item_id").alias("items"))
    )

    candidate_items = (
        train
        .group_by("item_id")
        .agg(pl.len().alias("count"))
        .sort(by=["count", "item_id"], descending=[True, False])
        .select("item_id")
    )

    benchmark_train = (
        train
        .sort(["uid", "timestamp"])
        .with_columns(
            (pl.col("timestamp") - pl.col("timestamp").min() + 1).alias("time_weight")
        )
    )

    user_histories_for_test = (
        benchmark_train
        .group_by("uid", maintain_order=True)
        .agg(pl.col("item_id").alias("items"))
    )

    catalog_size = train.get_column("item_id").n_unique()

    return {
        "data": data,
        "train": train,
        "benchmark_train": benchmark_train,
        "test": test,
        "test_targets_df": test_targets_df,
        "test_targets": dict(test_targets_df.iter_rows()),
        "test_users": test_targets_df.get_column("uid").to_list(),
        "candidate_items": candidate_items.get_column("item_id").to_list(),
        "user_histories_for_test": user_histories_for_test,
        "catalog_size": catalog_size,
    }

bundle = load_data()

train = bundle["train"]
benchmark_train = bundle["benchmark_train"]
test = bundle["test"]
test_targets_df = bundle["test_targets_df"]
test_targets = bundle["test_targets"]
test_users = bundle["test_users"]
candidate_items = bundle["candidate_items"]
user_histories_for_test = bundle["user_histories_for_test"]
catalog_size = bundle["catalog_size"]

print(f"train rows: {train.height:,}")
print(f"benchmark train rows: {benchmark_train.height:,}")
print(f"test rows: {test.height:,}")
print(f"test users: {len(test_users):,}")
print(f"candidate items: {len(candidate_items):,}")
print(f"catalog size for coverage: {catalog_size:,}")


train rows: 84,236,433
benchmark train rows: 84,236,433
test rows: 2,809,571
test users: 386,313
candidate items: 564,786
catalog size for coverage: 564,786


In [28]:
def get_metrics(targets: List[int], candidates: List[int], topk: int) -> Dict[str, float]:
    candidates = candidates[:topk]

    target_set = set(targets)
    denom = min(len(target_set), topk)

    hits = np.array([1.0 if item in target_set else 0.0 for item in candidates], dtype=np.float64)

    hitrate = float(hits.sum() > 0)
    recall = float(hits.sum() / denom) if denom > 0 else 0.0

    discounts = 1.0 / np.log2(np.arange(2, topk + 2, dtype=np.float64))
    dcg = float((hits * discounts[:len(hits)]).sum())
    idcg = float(discounts[:denom].sum())
    ndcg = dcg / idcg if idcg > 0 else 0.0

    return {"hitrate": hitrate, "recall": recall, "ndcg": ndcg}

def evaluate(
    targets: Dict[int, List[int]],
    candidates: Dict[int, List[int]],
    catalog_size: int,
    topk: int = TOPK,
) -> Dict[str, float]:
    metrics_sum = defaultdict(float)
    recommended_items = set()

    for uid, user_targets in targets.items():
        user_candidates = candidates.get(uid, [])[:topk]

        m = get_metrics(user_targets, user_candidates, topk=topk)
        for name, value in m.items():
            metrics_sum[name] += value

        recommended_items.update(user_candidates)

    num_users = len(targets)
    out = {name: value / num_users for name, value in metrics_sum.items()}
    out["coverage"] = len(recommended_items) / catalog_size if catalog_size > 0 else 0.0
    return out

def pred_df_to_candidates(pred_df: pl.DataFrame, test_users: List[int]) -> Dict[int, List[int]]:
    pred_map = dict(pred_df.select(["uid", "items"]).iter_rows())
    return {uid: pred_map.get(uid, []) for uid in test_users}

def score_predictions(model_name: str, pred_df: pl.DataFrame) -> pl.DataFrame:
    metrics = evaluate(
        targets=test_targets,
        candidates=pred_df_to_candidates(pred_df, test_users),
        catalog_size=catalog_size,
        topk=TOPK,
    )
    return pl.DataFrame([{"model": model_name, **metrics}])


## Random


In [29]:
def get_random_predict(candidate_items: List[int], test_users: List[int], k: int = TOPK, seed: int = 42) -> pl.DataFrame:
    replace = len(candidate_items) < k
    rng = np.random.default_rng(seed)
    shared_items = rng.choice(candidate_items, size=k, replace=replace).tolist()
    return pl.DataFrame({"uid": test_users, "items": [shared_items] * len(test_users)})

random_preds = get_random_predict(candidate_items, test_users, k=TOPK)
random_metrics = score_predictions("RandomModel", random_preds)
random_metrics


model,hitrate,recall,ndcg,coverage
str,f64,f64,f64,f64
"""RandomModel""",0.00036,0.000043,0.000014,0.000177


## TopPopular


In [30]:
def generate_top_popular_predict(
    train_data: pl.DataFrame,
    test_users: List[int],
    k: int = TOPK,
    recent_seconds: int | None = None,
) -> pl.DataFrame:
    source = train_data

    if recent_seconds is not None and train_data.height > 0:
        cutoff_ts = int(train_data.get_column("timestamp").max()) - recent_seconds
        recent_source = train_data.filter(pl.col("timestamp") >= cutoff_ts)
        if recent_source.height > 0:
            source = recent_source

    top_items = (
        source
        .group_by("item_id")
        .agg(pl.len().alias("count"))
        .sort(by=["count", "item_id"], descending=[True, False])
        .head(k)
        .get_column("item_id")
        .to_list()
    )

    return pl.DataFrame({"uid": test_users, "items": [top_items] * len(test_users)})

toppop_preds = generate_top_popular_predict(benchmark_train, test_users, k=TOPK)
toppop_metrics = score_predictions("PopularModel", toppop_preds)
toppop_metrics


model,hitrate,recall,ndcg,coverage
str,f64,f64,f64,f64
"""PopularModel""",0.110879,0.030551,0.010684,0.000177


In [31]:
toppop_recent_preds = generate_top_popular_predict(
    benchmark_train,
    test_users,
    k=TOPK,
    recent_seconds=TEST_INTERVAL_SECONDS,
)
toppop_recent_metrics = score_predictions("PopularModelLastWeek", toppop_recent_preds)
toppop_recent_metrics


model,hitrate,recall,ndcg,coverage
str,f64,f64,f64,f64
"""PopularModelLastWeek""",0.218597,0.071321,0.029366,0.000177


## HistoryLookUp


In [32]:
def generate_history_lookup_predict(
    histories_df: pl.DataFrame,
    test_users: List[int],
    k: int = TOPK,
) -> pl.DataFrame:
    preds = (
        histories_df
        .filter(pl.col("uid").is_in(test_users))
        .with_columns(pl.col("items").list.tail(k))
        .select(["uid", "items"])
    )
    return preds

history_preds = generate_history_lookup_predict(user_histories_for_test, test_users, k=TOPK)
history_metrics = score_predictions("HistoryModel", history_preds)
history_metrics


model,hitrate,recall,ndcg,coverage
str,f64,f64,f64,f64
"""HistoryModel""",0.076738,0.031291,0.010227,0.926151


## Item KNN & ALS


In [55]:
benchmark_train = benchmark_train.with_columns(pl.col('time_weight') / (3600 * 24)) 

In [56]:
def get_coo_matrix(
    df: pl.DataFrame,
    user_col: str = "uid",
    item_col: str = "item_id",
    weight_col: str | None = None,
    users_mapping: dict | None = None,
    items_mapping: dict | None = None,
):
    if weight_col is None:
        weights = np.ones(len(df), dtype=np.float32)
    else:
        weights = df.get_column(weight_col).to_numpy().astype(np.float32)

    row_ind = df.get_column(user_col).map_elements(users_mapping.get).to_numpy()
    col_ind = df.get_column(item_col).map_elements(items_mapping.get).to_numpy()

    return sp.coo_matrix((weights, (row_ind, col_ind)))

def predict_impl(
    model,
    test_users: List[int],
    mat,
    users_mapping: dict,
    items_inv_mapping: dict,
    n: int = TOPK,
    filter_already_liked_items: bool = False,
):
    recs = []

    for uid in tqdm(test_users):
        row_id = users_mapping.get(uid)
        if row_id is None:
            recs.append([])
            continue

        ranks = model.recommend(
            row_id,
            mat[row_id],
            N=n,
            filter_already_liked_items=filter_already_liked_items,
        )
        recs.append([items_inv_mapping[int(it)] for it in ranks[0]])

    return recs

users_list = benchmark_train.get_column("uid").unique().to_list()
items_list = benchmark_train.get_column("item_id").unique().to_list()

users_inv_mapping = dict(enumerate(users_list))
users_mapping = {v: k for k, v in users_inv_mapping.items()}

items_inv_mapping = dict(enumerate(items_list))
items_mapping = {v: k for k, v in items_inv_mapping.items()}

train_mat = get_coo_matrix(
    df=benchmark_train,
    user_col="uid",
    item_col="item_id",
    users_mapping=users_mapping,
    items_mapping=items_mapping,
).tocsr()

train_mat_time = get_coo_matrix(
    df=benchmark_train,
    user_col="uid",
    item_col="item_id",
    weight_col="time_weight",
    users_mapping=users_mapping,
    items_mapping=items_mapping,
).tocsr()


In [57]:
model_tfidf = TFIDFRecommender(K=128)
model_tfidf.fit(train_mat)

tfidf_recs = predict_impl(
    model_tfidf,
    test_users,
    train_mat,
    users_mapping,
    items_inv_mapping,
    n=TOPK,
    filter_already_liked_items=False,
)

tfidf_preds = pl.DataFrame({"uid": test_users, "items": tfidf_recs})
tfidf_metrics = score_predictions("TfIDFModel", tfidf_preds)
tfidf_metrics


/usr/local/lib/python3.12/dist-packages/implicit/utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.49638915061950684 seconds
  warnings.warn(


  0%|          | 0/564786 [00:00<?, ?it/s]

  0%|          | 0/386313 [00:00<?, ?it/s]

model,hitrate,recall,ndcg,coverage
str,f64,f64,f64,f64
"""TfIDFModel""",0.210889,0.068277,0.022775,0.979118


In [58]:
model_tfidf_time = TFIDFRecommender(K=128)
model_tfidf_time.fit(train_mat_time)

tfidf_time_recs = predict_impl(
    model_tfidf_time,
    test_users,
    train_mat_time,
    users_mapping,
    items_inv_mapping,
    n=TOPK,
    filter_already_liked_items=False,
)

tfidf_time_preds = pl.DataFrame({"uid": test_users, "items": tfidf_time_recs})
tfidf_time_metrics = score_predictions("TfIDFModelTimeW", tfidf_time_preds)
tfidf_time_metrics


/usr/local/lib/python3.12/dist-packages/implicit/utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.4901692867279053 seconds
  warnings.warn(


  0%|          | 0/564786 [00:00<?, ?it/s]

  0%|          | 0/386313 [00:00<?, ?it/s]

model,hitrate,recall,ndcg,coverage
str,f64,f64,f64,f64
"""TfIDFModelTimeW""",0.238296,0.079488,0.026717,0.984904


In [59]:
model_als = AlternatingLeastSquares(
    factors=256,
    iterations=32,
    calculate_training_loss=False,
    regularization=0.1,
)

model_als.fit(train_mat)

als_recs = predict_impl(
    model_als,
    test_users,
    train_mat,
    users_mapping,
    items_inv_mapping,
    n=TOPK,
    filter_already_liked_items=False,
)

als_preds = pl.DataFrame({"uid": test_users, "items": als_recs})
als_metrics = score_predictions("ALSModel", als_preds)
als_metrics


  0%|          | 0/32 [00:00<?, ?it/s]

  0%|          | 0/386313 [00:00<?, ?it/s]

model,hitrate,recall,ndcg,coverage
str,f64,f64,f64,f64
"""ALSModel""",0.232565,0.070308,0.023201,0.027357


In [60]:
model_als_time = AlternatingLeastSquares(
    factors=256,
    iterations=32,
    calculate_training_loss=False,
    regularization=0.1,
)

model_als_time.fit(train_mat_time)

als_time_recs = predict_impl(
    model_als_time,
    test_users,
    train_mat_time,
    users_mapping,
    items_inv_mapping,
    n=TOPK,
    filter_already_liked_items=False,
)

als_time_preds = pl.DataFrame({"uid": test_users, "items": als_time_recs})
als_time_metrics = score_predictions("ALSModelTimeW", als_time_preds)
als_time_metrics


  0%|          | 0/32 [00:00<?, ?it/s]

  0%|          | 0/386313 [00:00<?, ?it/s]

model,hitrate,recall,ndcg,coverage
str,f64,f64,f64,f64
"""ALSModelTimeW""",0.294282,0.092734,0.03294,0.173699


## Markov Model


In [61]:
class MarkovChainTopNext:
    def __init__(self, topk: int = TOPK):
        self.topk = topk
        self.next_counts = defaultdict(Counter)
        self.top_next = {}

    def fit(self, sequences: List[List[int]]):
        nc = self.next_counts
        for seq in tqdm(sequences):
            if seq is None or len(seq) < 2:
                continue
            for a, b in zip(seq[:-1], seq[1:]):
                nc[a][b] += 1

        top_next = {}
        for a, counter in nc.items():
            items = list(counter.items())
            items.sort(key=lambda x: (-x[1], str(x[0])))
            top_next[a] = [x[0] for x in items[:self.topk]]

        self.top_next = top_next
        return self

    def predict_next_batch(self, items: List[int], k: int | None = None):
        if k is None:
            k = self.topk
        return [self.top_next.get(it, [])[:k] for it in items]

train_sequences = (
    benchmark_train
    .group_by("uid", maintain_order=True)
    .agg(pl.col("item_id").alias("item_seq"))
    .get_column("item_seq")
    .to_list()
)

markov_model = MarkovChainTopNext(topk=TOPK)
markov_model.fit(train_sequences)


  0%|          | 0/817599 [00:00<?, ?it/s]

In [62]:
test_last_df = (
    user_histories_for_test
    .filter(pl.col("uid").is_in(test_users))
    .with_columns(pl.col("items").list.tail(1).alias("last_item"))
    .select(["uid", "last_item"])
)

uids_for_markov = test_last_df.get_column("uid").to_list()
last_items = [
    items[0] if items is not None and len(items) > 0 else None
    for items in test_last_df.get_column("last_item").to_list()
]

preds = [
    markov_model.predict_next_batch([item], k=TOPK)[0] if item is not None else []
    for item in last_items
]

markov_preds = pl.DataFrame({"uid": uids_for_markov, "items": preds})
markov_metrics = score_predictions("MarkovChain", markov_preds)
markov_metrics


model,hitrate,recall,ndcg,coverage
str,f64,f64,f64,f64
"""MarkovChain""",0.188026,0.057564,0.024964,0.730514


## Total DataFrame


In [63]:
results = pl.concat([
    random_metrics,
    toppop_metrics,
    toppop_recent_metrics,
    history_metrics,
    tfidf_metrics,
    tfidf_time_metrics,
    als_metrics,
    als_time_metrics,
    markov_metrics,
]).sort("ndcg", descending=True)

results


model,hitrate,recall,ndcg,coverage
str,f64,f64,f64,f64
"""ALSModelTimeW""",0.294282,0.092734,0.03294,0.173699
"""PopularModelLastWeek""",0.218597,0.071321,0.029366,0.000177
"""TfIDFModelTimeW""",0.238296,0.079488,0.026717,0.984904
"""MarkovChain""",0.188026,0.057564,0.024964,0.730514
"""ALSModel""",0.232565,0.070308,0.023201,0.027357
"""TfIDFModel""",0.210889,0.068277,0.022775,0.979118
"""PopularModel""",0.110879,0.030551,0.010684,0.000177
"""HistoryModel""",0.076738,0.031291,0.010227,0.926151
"""RandomModel""",0.00036,0.000043,0.000014,0.000177
